In [1]:
# ============================================================
# Cell 1: Setup
# 16_naked_atm_put_incremental_update_and_intraday_v0_1
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

cwd = Path.cwd()

if cwd.name.lower() == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"
PRODUCTION_DIR = DATA_DIR / "production"

THETADATA_CHAIN_DIR = RAW_DATA_DIR / "thetadata_chains"

for d in [RAW_DATA_DIR, PROCESSED_DATA_DIR, AUDIT_DIR, PRODUCTION_DIR, THETADATA_CHAIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 250)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("THETADATA_CHAIN_DIR:", THETADATA_CHAIN_DIR)

PROJECT_ROOT: C:\Users\patri\vrp_project
THETADATA_CHAIN_DIR: C:\Users\patri\vrp_project\data\raw\thetadata_chains


In [2]:
# ============================================================
# Cell 2: Production file paths and constants
# ============================================================

NAKED_ATM_EOD_PANEL = (
    PROCESSED_DATA_DIR / "naked_atm_put_eod_panel_v0_1.parquet"
)

NAKED_ATM_INTRADAY_SNAPSHOT = (
    PROCESSED_DATA_DIR / "naked_atm_put_intraday_snapshot_v0_1.parquet"
)

NAKED_ATM_SIGNAL_PANEL = (
    PROCESSED_DATA_DIR / "naked_atm_put_production_signal_panel_v0_1.parquet"
)

TARGET_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

assert NAKED_ATM_EOD_PANEL.exists(), NAKED_ATM_EOD_PANEL

print("EOD panel:", NAKED_ATM_EOD_PANEL)
print("Intraday snapshot:", NAKED_ATM_INTRADAY_SNAPSHOT)
print("Signal panel:", NAKED_ATM_SIGNAL_PANEL)
print("Target tenors:", TARGET_TENORS)

EOD panel: C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet
Intraday snapshot: C:\Users\patri\vrp_project\data\processed\naked_atm_put_intraday_snapshot_v0_1.parquet
Signal panel: C:\Users\patri\vrp_project\data\processed\naked_atm_put_production_signal_panel_v0_1.parquet
Target tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]


In [3]:
# ============================================================
# Cell 3: Load current EOD panel
# ============================================================

eod_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()

eod_panel["trade_dt"] = pd.to_datetime(eod_panel["trade_dt"]).dt.normalize()
eod_panel["entry_tenor"] = eod_panel["entry_tenor"].astype(int)

latest_panel_date = eod_panel["trade_dt"].max()

print("Current EOD panel:")
print("Rows:", len(eod_panel))
print("Start:", eod_panel["trade_dt"].min().date())
print("End:", latest_panel_date.date())
print("Unique dates:", eod_panel["trade_dt"].nunique())
print("Unique tenors:", sorted(eod_panel["entry_tenor"].unique().tolist()))

display(eod_panel.tail(9))

Current EOD panel:
Rows: 18099
Start: 2018-06-25
End: 2026-06-25
Unique dates: 2011
Unique tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]


,trade_dt,quote_timestamp,entry_tenor,expiry,actual_dte,underlying_price,atm_strike,atm_put_bid,atm_put_ask,atm_put_mid,atm_put_iv,atm_put_delta,tenor_group,vix_style_vol,implied_variance,spx_rsi_14,entry_rsi_14,old_vrp_signal,old_vrp_3m_z,old_vrp_1y_z,old_vrp_blended_z,simple_carry_vrp_signal,simple_carry_vrp_signal_z_3m,simple_carry_vrp_signal_z_1y,simple_carry_vrp_min_z,trailing_carry_vrp_signal,trailing_carry_vrp_signal_z_3m,trailing_carry_vrp_signal_z_1y,trailing_carry_vrp_min_z,expiry_spx_close,expiry_intrinsic_value,expired_otm,entry_credit_points,short_option_pnl_points,normalized_pnl_dollars,test_pnl,win,test_win,pricing_status,mapping_status,selected_chain_file,cache_status,selected_chain_file_live,put_moneyness,put_pct_otm
18090,2026-06-25,2026-06-25T16:00:00.000,9,2026-07-02,7,7357.49,7355.0,59.9,61.5,60.70,NaN,NaN,front_9_15d,17.952376,0.032229,46.297996,46.297996,0.483909,0.114250,-0.395434,-0.140592,0.773380,0.442191,-0.190450,-0.190450,0.483909,0.128162,-0.395629,-0.395629,NaN,NaN,None,59.9,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260702_160000.pkl,0.999662,0.000338
18091,2026-06-25,2026-06-25T16:00:00.000,12,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,NaN,NaN,front_9_15d,17.964763,0.032273,46.297996,46.297996,0.293640,-0.378609,-0.842333,-0.610471,0.683950,0.156820,-0.507614,-0.507614,0.293640,-0.359177,-0.840252,-0.840252,NaN,NaN,None,84.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,0.999662,0.000338
18092,2026-06-25,2026-06-25T16:00:00.000,15,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,NaN,NaN,front_9_15d,17.972192,0.032300,46.297996,46.297996,0.171854,-0.531361,-1.088037,-0.809699,0.509373,-0.115243,-0.801644,-0.801644,0.171854,-0.509206,-1.082160,-1.082160,NaN,NaN,None,84.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,0.999662,0.000338
18093,2026-06-25,2026-06-25T16:00:00.000,18,2026-07-10,15,7357.49,7355.0,84.8,86.2,85.50,NaN,NaN,middle_18_24d,17.707876,0.031357,46.297996,46.297996,0.099408,-0.773276,-1.333368,-1.053322,0.540072,-0.146100,-0.843356,-0.843356,0.099408,-0.747389,-1.323083,-1.323083,NaN,NaN,None,84.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPXW_20260625_20260710_160000.pkl,0.999662,0.000338
18094,2026-06-25,2026-06-25T16:00:00.000,21,2026-07-17,22,7357.49,7355.0,100.8,101.9,101.35,NaN,NaN,middle_18_24d,17.516638,0.030683,46.297996,46.297996,-0.182296,-1.505797,-1.955464,-1.730630,0.277368,-0.476921,-1.148563,-1.148563,-0.182296,-1.456025,-1.932966,-1.932966,NaN,NaN,None,100.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,0.999662,0.000338
18095,2026-06-25,2026-06-25T16:00:00.000,24,2026-07-17,22,7357.49,7355.0,100.8,101.9,101.35,NaN,NaN,middle_18_24d,18.028765,0.032504,46.297996,46.297996,-0.024917,-1.193720,-1.737542,-1.465631,0.399541,-0.201362,-0.928478,-0.928478,-0.024917,-1.158431,-1.719697,-1.719697,NaN,NaN,None,100.8,NaN,NaN,NaN,None,False,not_priced,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,mapped,C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260625_20260717_160000.pkl,0.999662,0.000338
18096,2026-06-25,2026-06-25T16:00:00.000,27,2026-07-24,29,7357.49,7350.0,115.6,116.8,116.20,NaN,NaN,back_27_33

In [4]:
# ============================================================
# Cell 4: Dynamic EOD backfill window and intraday run date
# ============================================================

from datetime import time
from zoneinfo import ZoneInfo

# Normal daily use:
#   AUTO mode figures out the latest completed EOD date.
#
# Logic:
#   - Before EOD data is available: latest completed EOD = prior trading day
#   - After EOD data is available: latest completed EOD = today, if today is a trading day
#
# Adjust this if ThetaData EOD files are reliably ready earlier/later.
EOD_AVAILABLE_AFTER_ET = time(18, 30)

NY_TZ = ZoneInfo("America/New_York")
now_et = pd.Timestamp.now(tz=NY_TZ)

today_et = now_et.normalize().tz_localize(None)
current_time_et = now_et.time()

# Use NYSE calendar if available. Fall back to business days.
def get_trading_days(start_date, end_date):
    start_date = pd.Timestamp(start_date).normalize()
    end_date = pd.Timestamp(end_date).normalize()

    try:
        import pandas_market_calendars as mcal

        nyse = mcal.get_calendar("NYSE")
        schedule = nyse.schedule(
            start_date=start_date.strftime("%Y-%m-%d"),
            end_date=end_date.strftime("%Y-%m-%d"),
        )

        sessions = (
            pd.Series(schedule.index)
            .dt.tz_localize(None)
            .dt.normalize()
            .sort_values()
            .reset_index(drop=True)
        )

        calendar_source = "pandas_market_calendars_NYSE"

    except Exception:
        sessions = (
            pd.Series(pd.bdate_range(start_date, end_date))
            .dt.normalize()
            .sort_values()
            .reset_index(drop=True)
        )

        calendar_source = "pandas_bdate_range_fallback"

    return sessions, calendar_source


# Reload current EOD panel.
eod_panel = pd.read_parquet(NAKED_ATM_EOD_PANEL).copy()
eod_panel["trade_dt"] = pd.to_datetime(eod_panel["trade_dt"]).dt.normalize()
eod_panel["entry_tenor"] = eod_panel["entry_tenor"].astype(int)

latest_panel_date = eod_panel["trade_dt"].max()

# Build trading calendar from the day after the current panel through today.
future_trading_days, calendar_source = get_trading_days(
    latest_panel_date + pd.Timedelta(days=1),
    today_et,
)

if len(future_trading_days) == 0:
    latest_completed_eod_date = latest_panel_date
else:
    today_is_trading_day = today_et in set(future_trading_days)

    if today_is_trading_day and current_time_et >= EOD_AVAILABLE_AFTER_ET:
        latest_completed_eod_date = today_et
    else:
        prior_sessions = future_trading_days[future_trading_days < today_et]

        if len(prior_sessions) > 0:
            latest_completed_eod_date = prior_sessions.max()
        else:
            latest_completed_eod_date = latest_panel_date

intraday_run_date = today_et

# EOD dates to permanently add to history.
eod_update_dates = future_trading_days[
    (future_trading_days > latest_panel_date)
    & (future_trading_days <= latest_completed_eod_date)
]

eod_update_dates_df = pd.DataFrame({
    "eod_update_date": eod_update_dates,
})

print("Now ET:", now_et)
print("Calendar source:", calendar_source)
print("Latest panel date:", latest_panel_date.date())
print("Latest completed EOD date:", latest_completed_eod_date.date())
print("Intraday run date:", intraday_run_date.date())
print("EOD dates to add:", len(eod_update_dates_df))

display(eod_update_dates_df)

Now ET: 2026-07-01 21:08:39.308769-04:00
Calendar source: pandas_market_calendars_NYSE
Latest panel date: 2026-06-25
Latest completed EOD date: 2026-07-01
Intraday run date: 2026-07-01
EOD dates to add: 4


,eod_update_date
0,2026-06-26
1,2026-06-29
2,2026-06-30
3,2026-07-01


In [5]:
# ============================================================
# Cell 5: Check existing ThetaData cache for EOD update dates
# ============================================================

chain_files = sorted(THETADATA_CHAIN_DIR.glob("*.pkl"))

cache_rows = []

for path in chain_files:
    parts = path.stem.split("_")

    if len(parts) < 4:
        continue

    root = parts[0]
    trade_date_raw = parts[1]
    expiry_raw = parts[2]
    quote_time_raw = parts[3]

    trade_dt = pd.to_datetime(trade_date_raw, format="%Y%m%d", errors="coerce")
    expiry = pd.to_datetime(expiry_raw, format="%Y%m%d", errors="coerce")

    cache_rows.append({
        "root": root,
        "trade_dt": trade_dt,
        "expiry": expiry,
        "quote_time": quote_time_raw,
        "file": path.name,
        "path": str(path),
        "size_mb": path.stat().st_size / 1_000_000,
    })

chain_cache_inventory = pd.DataFrame(cache_rows)

update_dates_set = set(pd.to_datetime(eod_update_dates_df["eod_update_date"]).dt.normalize())

update_date_cache = chain_cache_inventory[
    chain_cache_inventory["trade_dt"].isin(update_dates_set)
].copy()

print("Total cached chain files:", len(chain_cache_inventory))
print("Cached chain date range:", chain_cache_inventory["trade_dt"].min(), "to", chain_cache_inventory["trade_dt"].max())
print("EOD update dates needed:", len(update_dates_set))
print("Cached files for update dates:", len(update_date_cache))

if len(update_date_cache) > 0:
    cache_summary = (
        update_date_cache
        .groupby("trade_dt")
        .agg(
            chain_files=("file", "count"),
            expiries=("expiry", lambda x: sorted(pd.to_datetime(x).dt.date.unique().tolist())),
            roots=("root", lambda x: sorted(set(x))),
            quote_times=("quote_time", lambda x: sorted(set(x))),
        )
        .reset_index()
    )

    display(cache_summary)
    display(update_date_cache.sort_values(["trade_dt", "expiry", "root", "quote_time"]))
else:
    print("No cached chain files found for the EOD update dates.")

Total cached chain files: 10901
Cached chain date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
EOD update dates needed: 4
Cached files for update dates: 0
No cached chain files found for the EOD update dates.


In [6]:
# ============================================================
# Cell 6: Update status and next production dependency
# ============================================================

# This notebook's job so far:
#   1. Load the current naked ATM put EOD panel.
#   2. Dynamically identify completed EOD dates missing after the panel end.
#   3. Check whether chain cache files already exist for those dates.
#
# Next step:
#   Run the existing VIX-style term-structure updater from Notebook 01
#   for eod_update_dates.
#
# Do not build a separate ThetaData downloader here.
# The VIX updater is the source-of-truth pipeline for:
#   - expiry mapping
#   - full SPX/SPXW chain cache creation
#   - 16:00 market-close snapshots
#   - VIX-style implied variance / term-structure history

update_status = {
    "latest_panel_date": latest_panel_date.date(),
    "latest_completed_eod_date": latest_completed_eod_date.date(),
    "intraday_run_date": intraday_run_date.date(),
    "eod_dates_to_add": [
        pd.Timestamp(x).date()
        for x in eod_update_dates_df["eod_update_date"].tolist()
    ],
    "cached_files_for_update_dates": int(len(update_date_cache)),
    "next_step": "Run existing VIX-style term-structure updater for eod_update_dates.",
}

display(pd.DataFrame([update_status]))


,latest_panel_date,latest_completed_eod_date,intraday_run_date,eod_dates_to_add,cached_files_for_update_dates,next_step
0,2026-06-25,2026-07-01,2026-07-01,"[2026-06-26, 2026-06-29, 2026-06-30, 2026-07-01]",0,Run existing VIX-style term-structure updater for eod_update_dates.
